In [12]:
"""
TASA Silver NAICS/ISIC Emissions Dashboard
Ingests TASA_Silver_NAICS_ISIC.csv and renders an interactive Dash dashboard
with stacked bar charts (Scope 1/2/3) and a 2020-2024 trend view.

Requirements:
    pip install pandas dash plotly
"""
#!pip install panel plotly
#!pip install ipywidgets plotly
!pip install jupyter_bokeh

import pandas as pd
import numpy as np
import plotly.graph_objects as go



   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   --------------------------------------- 914.9/914.9 kB 13.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 41.2 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.6
    Uninstalling widgetsnbextension-3.6.6:
      Successfully uninstalled widgetsnbextension-3.6.6
  Attempting uninstall: jupyterlab_widgets
    Found existing installation: jupyterlab-widgets 1.0.0
    Uninstalling jupyterlab-widgets-1.0.0:
      Successfully uninstalled jupyterlab-widgets-1.0.0
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.8.1
    Uninstalling ipywidgets-7.8.1:
      Successfully uninstalled ipywidgets-7.8.1


1. Import silver file

In [6]:
def load_data(path: str = "TASA_Silver_NAICS_ISIC.csv") -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)

    # Normalise column names (strip whitespace)
    df.columns = df.columns.str.strip()

    # Cast numeric columns
    numeric_cols = [
        "Base Year Total Emissions Intensity",
        "Base Year Scope 1", "Base Year Scope 2", "Base Year Scope 3",
        "2020 Total Emissions Intensity",
        "2020 Scope 1", "2020 Scope 2", "2020 Scope 3",
        "2023 Total Emissions Intensity",
        "2023 Scope 1", "2023 Scope 2", "2023 Scope 3",
        "2024 Total Emissions Intensity",
        "2024 Scope 1", "2024 Scope 2", "2024 Scope 3",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    #print(df.head())

    return df



df = load_data()

2. Convert years to long

In [ ]:
def to_long(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [
        "Base Year Total Emissions Intensity",
        "Base Year Scope 1", "Base Year Scope 2", "Base Year Scope 3",
        "2020 Total Emissions Intensity",
        "2020 Scope 1", "2020 Scope 2", "2020 Scope 3",
        "2023 Total Emissions Intensity",
        "2023 Scope 1", "2023 Scope 2", "2023 Scope 3",
        "2024 Total Emissions Intensity",
        "2024 Scope 1", "2024 Scope 2", "2024 Scope 3",
    ]

    id_cols = [c for c in df.columns if c not in numeric_cols]

    df_long = df.melt(id_vars=id_cols, value_vars=numeric_cols,
                      var_name="raw_col", value_name="value")

    def parse_year_scope(col_name):
        if col_name.startswith("Base Year"):
            return "Base Year", col_name.replace("Base Year ", "")
        parts = col_name.split(" ", 1)
        return parts[0], parts[1]

    df_long[["year", "scope"]] = df_long["raw_col"].apply(
        lambda x: pd.Series(parse_year_scope(x))
    )
    #print(df_long.head())
    return df_long.drop(columns=["raw_col"])

df_long = to_long(df)


In [17]:
df_long.head()


,Country Code,Country Name,NAICS Code,NAICS Definition,ISIC Rev. 4 Code,ISIC Rev. 4 Numeric Code,ISIC Rev. 4 Title,Country Sector,value,year,scope
0,AFG,Afghanistan,111110,Soybean Farming,A0111*,0111,"Growing of cereals (except rice), leguminous c...",Beans and Nuts,603.696195,Base Year,Total Emissions Intensity
1,AFG,Afghanistan,111120,Oilseed (except Soybean) Farming,A0111*,0111,"Growing of cereals (except rice), leguminous c...",Vegetables,609.629727,Base Year,Total Emissions Intensity
2,AFG,Afghanistan,111130,Dry Pea and Bean Farming,A0111*,0111,"Growing of cereals (except rice), leguminous c...",Beans and Nuts,603.696195,Base Year,Total Emissions Intensity
3,AFG,Afghanistan,111140,Wheat Farming,A0111*,0111,"Growing of cereals (except rice), leguminous c...",Other Cereals,262.572715,Base Year,Total Emissions Intensity
4,AFG,Afghanistan,111150,Corn Farming,A0111*,0111,"Growing of cereals (except rice), leguminous c...",Maize,684.916159,Base Year,Total Emissions Intensity


3. Make some graphs

In [43]:
print(df_long.columns.tolist())
print(df_long[df_long["ISIC Rev. 4 Title"] == selected_isic[0]]["Country Name"].unique())
print(df_long.groupby(["ISIC Rev. 4 Title", "Country Name"])["Country Sector"].nunique().head(20))

['Country Code', 'Country Name', 'NAICS Code', 'NAICS Definition', 'ISIC Rev. 4 Code', 'ISIC Rev. 4 Numeric Code', 'ISIC Rev. 4 Title', 'Country Sector', 'value', 'year', 'scope']
['Afghanistan' 'Angola' 'Albania' 'United Arab Emirates' 'Argentina'
 'Armenia' 'Australia' 'Austria' 'Azerbaijan' 'Burundi' 'Belgium' 'Benin'
 'Burkina Faso' 'Bangladesh' 'Bulgaria' 'Bahrain' 'Bahamas'
 'Bosnia and Herzegovina' 'Belarus' 'Belize' 'Bolivia' 'Brazil' 'Brunei'
 'Bhutan' 'Botswana' 'Canada' 'Switzerland' 'Chile' 'China'
 "Côte d'Ivoire" 'Cameroon' 'Democratic Republic of Congo'
 'Republic of Congo' 'Colombia' 'Costa Rica' 'Cyprus' 'Czech Republic'
 'Germany' 'Djibouti' 'Denmark' 'Dominican Republic' 'Algeria' 'Ecuador'
 'Egypt' 'Eritrea' 'Spain' 'Estonia' 'Ethiopia' 'Finland' 'France' 'Gabon'
 'United Kingdom' 'Georgia' 'Ghana' 'Guinea' 'Gambia' 'Equatorial Guinea'
 'Greece' 'Guatemala' 'Honduras' 'Croatia' 'Haiti' 'Hungary' 'Indonesia'
 'India' 'Ireland' 'Iran' 'Iraq' 'Iceland' 'Israel' 'Italy'

In [60]:
import plotly.graph_objects as go
from ipywidgets import widgets
from IPython.display import display, HTML
import pandas as pd
import textwrap

df_long["year"] = df_long["year"].astype(str)

SCOPE_COLORS = {"Scope 1": "#e8534a", "Scope 2": "#f0a500", "Scope 3": "#4a90d9"}
GAP_SECTOR  = 0.6
GAP_COUNTRY = 1.8

all_isic      = sorted(df_long["ISIC Rev. 4 Title"].unique())
all_countries = sorted(df_long["Country Name"].unique())
all_years     = ["2020", "2023", "2024"]

# ── Styles + header ──
display(HTML("""
<style>
  .widget-dropdown select, .widget-select select {
    background: #121522 !important; color: #c8cde0 !important;
    border: 1.5px solid #ffffff !important; border-radius: 8px !important;
    font-family: 'DM Sans', sans-serif !important; font-size: 12px !important;
    padding: 4px 8px !important;
  }
  .widget-dropdown select:focus, .widget-select select:focus {
    border-color: #ffffff !important; outline: none !important;
  }
  .widget-select select option:checked { background: #1e2a4a !important; color: #7bb8f5 !important; }
  .jp-Cell-outputArea, .jp-OutputArea-output, div.output_area { background: #0d0f1a !important; }
</style>
"""))

display(HTML("""
<div style='font-family:"DM Sans","Segoe UI",sans-serif; background:#0d0f1a;
            border:1px solid #ffffff; padding:20px 24px 16px;
            margin-bottom:16px; border-radius:12px;'>
  <p style='font-size:22px;font-weight:800;color:#e8ecff;letter-spacing:-0.02em;margin:0 0 4px'>
    Scope Emissions · Country Sector View</p>
  <p style='font-size:12px;color:#4a5070;margin:0'>
    Scope 1 · 2 · 3 emissions intensity by ISIC sector, country, and year</p>
</div>
"""))

# ── Widgets ──
def ctrl_label(text):
    return widgets.HTML(
        f'<div style="font-size:10px;font-weight:700;letter-spacing:0.12em;color:#4a90d9;'
        f'text-transform:uppercase;margin-bottom:6px;font-family:DM Sans,sans-serif">{text}</div>'
    )

isic_w    = widgets.Dropdown(options=all_isic, layout=widgets.Layout(width="520px"))
country_w = widgets.SelectMultiple(options=all_countries, value=["Australia", "Brazil", "Germany"],
                                   rows=7, layout=widgets.Layout(width="220px"))
year_w    = widgets.SelectMultiple(options=all_years, value=all_years,
                                   rows=3, layout=widgets.Layout(width="120px"))
hint      = widgets.HTML(
    '<div style="font-size:11px;color:#3a4060;font-family:DM Sans,sans-serif;margin-top:6px">'
    'Hold Ctrl / Cmd to select multiple</div>'
)

controls = widgets.HBox([
    widgets.VBox([ctrl_label("ISIC Sector"), isic_w]),
    widgets.VBox([ctrl_label("Countries"), country_w, hint]),
    widgets.VBox([ctrl_label("Years"), year_w]),
], layout=widgets.Layout(
    background='#0d0f1a',
    border='1px solid #ffffff',
    padding='16px 20px',
    margin='0 0 12px',
    border_radius='10px',
    gap='24px',
))

out = widgets.Output()

# ── Helpers ──
def wrap_label(name, width=20):
    return "<br>".join(textwrap.wrap(str(name), width=width))

# ── Chart builder ──
def build_chart(isic, countries, years):
    filtered = df_long[
        (df_long["ISIC Rev. 4 Title"] == isic) &
        (df_long["Country Name"].isin(countries)) &
        (df_long["year"].isin(years))
    ].copy()

    if filtered.empty:
        return None

    agg = (
        filtered
        .groupby(["Country Name", "Country Sector", "year", "scope"])["value"]
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=["Scope 1", "Scope 2", "Scope 3"], fill_value=0)
        .reset_index()
    )
    agg["Total"] = agg[["Scope 1", "Scope 2", "Scope 3"]].sum(axis=1)
    agg = agg.sort_values(["Country Name", "Country Sector", "year"])

    # ── Build x positions ──
    x_positions, tickvals, ticktext = [], [], []
    sector_label_pos  = {}
    country_label_pos = {}
    pos = 0

    for country in countries:
        if country not in agg["Country Name"].values:
            continue
        country_start = pos
        sectors = sorted(agg[agg["Country Name"] == country]["Country Sector"].unique())

        for sector in sectors:
            sector_start = pos
            rows = agg[
                (agg["Country Name"] == country) &
                (agg["Country Sector"] == sector)
            ].sort_values("year")
            for _, row in rows.iterrows():
                x_positions.append(pos)
                tickvals.append(pos)
                ticktext.append(row["year"])
                pos += 1
            sector_label_pos[(country, sector)] = (sector_start + pos - 1) / 2
            pos += GAP_SECTOR

        country_label_pos[country] = (country_start + pos - GAP_SECTOR) / 2
        pos += GAP_COUNTRY

    # ── Map positions back to df ──
    x_pos_map = {}
    xi = 0
    for country in countries:
        if country not in agg["Country Name"].values:
            continue
        for sector in sorted(agg[agg["Country Name"] == country]["Country Sector"].unique()):
            for year in sorted(years):
                mask = (
                    (agg["Country Name"] == country) &
                    (agg["Country Sector"] == sector) &
                    (agg["year"] == year)
                )
                if mask.any():
                    x_pos_map[(country, sector, year)] = x_positions[xi]
                    xi += 1

    agg["x_pos"] = agg.apply(
        lambda r: x_pos_map.get((r["Country Name"], r["Country Sector"], r["year"])), axis=1
    )
    agg = agg.dropna(subset=["x_pos"])

    # ── Traces ──
    fig = go.Figure()
    for scope in ["Scope 1", "Scope 2", "Scope 3"]:
        fig.add_trace(go.Bar(
            name=scope,
            x=agg["x_pos"],
            y=agg[scope],
            marker_color=SCOPE_COLORS[scope],
            marker_line_width=0,
            marker_opacity=0.92,
            width=0.68,
            customdata=agg[["Total","Scope 1","Scope 2","Scope 3",
                             "Country Sector","Country Name","year"]].values,
            hovertemplate=(
                "<b style='font-size:13px'>%{customdata[5]}</b><br>"
                "<span style='color:#8a90a8'>%{customdata[4]} · %{customdata[6]}</span><br><br>"
                "<b>Total: %{customdata[0]:,.1f}</b><br>"
                "<span style='color:#3a4060'>──────────────</span><br>"
                "<span style='color:#e8534a'>▮</span> Scope 1 &nbsp; %{customdata[1]:,.1f}<br>"
                "<span style='color:#f0a500'>▮</span> Scope 2 &nbsp; %{customdata[2]:,.1f}<br>"
                "<span style='color:#4a90d9'>▮</span> Scope 3 &nbsp; %{customdata[3]:,.1f}<br>"
                "<extra></extra>"
            ),
        ))

    # ── Annotations ──
    annotations = []

    for (country, sector), mid in sector_label_pos.items():
        annotations.append(dict(
            x=mid, y=-0.34, xref="x", yref="paper",
            text=wrap_label(sector, width=16),
            showarrow=False,
            font=dict(size=10, color="#5a6080", family="DM Sans, sans-serif"),
            xanchor="center", align="center",
        ))

    for country, mid in country_label_pos.items():
        annotations.append(dict(
            x=mid, y=-0.58, xref="x", yref="paper",
            text=f"<b>{country}</b>",
            showarrow=False,
            font=dict(size=13, color="#7bb8f5", family="DM Sans, sans-serif"),
            xanchor="center",
        ))

    annotations.append(dict(
        x=0.5, y=1.07, xref="paper", yref="paper",
        text=f"<span style='color:#4a90d9'>▸</span>  {isic}",
        showarrow=False,
        font=dict(size=11, color="#5a6080", family="DM Sans, sans-serif"),
        xanchor="center", align="center",
    ))

    # ── Separators ──
    max_y = agg[["Scope 1","Scope 2","Scope 3"]].sum(axis=1).max() * 1.05
    valid = [c for c in countries if c in agg["Country Name"].values]
    for country in valid[:-1]:
        last_pos = agg[agg["Country Name"] == country]["x_pos"].max()
        sep = last_pos + GAP_SECTOR / 2 + GAP_COUNTRY / 3
        fig.add_shape(type="line", x0=sep, x1=sep, y0=0, y1=max_y,
                      line=dict(color="#2a3050", width=1, dash="dot"))

    fig.update_layout(
        barmode="stack",
        paper_bgcolor="#0d0f1a",
        plot_bgcolor="#0d0f1a",
        font=dict(family="DM Sans, Segoe UI, sans-serif", color="#c8cde0", size=12),
        hoverlabel=dict(
            bgcolor="#151827", bordercolor="#2e3249",
            font=dict(size=12, color="#e8ecff", family="DM Sans, sans-serif"),
            namelength=0,
        ),
        xaxis=dict(
            tickvals=tickvals, ticktext=ticktext,
            tickfont=dict(size=10, color="#5a6080"),
            tickangle=0, showgrid=False, zeroline=False,
            linecolor="#1e2235",
            range=[-0.8, max(x_positions) + 0.8] if x_positions else [0, 1],
        ),
        yaxis=dict(
            gridcolor="#151c2e", gridwidth=1, zeroline=False,
            linecolor="#1e2235", title="Emissions Intensity",
            titlefont=dict(color="#4a5070", size=11),
            tickfont=dict(color="#5a6080", size=10),
        ),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02,
            xanchor="right", x=1, bgcolor="rgba(0,0,0,0)",
            font=dict(size=11, color="#8a90a8"),
        ),
        annotations=annotations,
        margin=dict(t=90, b=200, l=70, r=30),
        width=1200,   # fixed width always
        height=580,
        bargap=0.12,
    )
    return fig

# ── Render ──
def on_change(_):
    with out:
        out.clear_output(wait=True)
        fig = build_chart(isic_w.value, list(country_w.value), list(year_w.value))
        if fig is None:
            display(HTML("""
            <div style='padding:40px;text-align:center;color:#3a4060;
                        font-family:DM Sans,sans-serif;font-size:14px;
                        background:#0d0f1a;border-radius:10px;border:1px dashed #ffffff'>
              No data for this selection — try different countries or ISIC sector
            </div>"""))
        else:
            fig.show()

isic_w.observe(on_change, names="value")
country_w.observe(on_change, names="value")
year_w.observe(on_change, names="value")

scroll_box = widgets.Box(
    [out],
    layout=widgets.Layout(
        overflow_x="auto",
        width="100%",
        border="1px solid #ffffff",
        border_radius="10px",
    )
)

display(controls, scroll_box)
on_change(None)

Box(children=(Output(),), layout=Layout(border='1px solid #ffffff', overflow_x='auto', width='100%'))